# Sharing the solved bus power between generators

When several synchronous generators sit on one bus the power flow solves for
the bus as a whole. This notebook checks how that bus total is written back to
the individual machines, at a slack bus and at a PV bus, and what happens when
a machine is too small to take its share.


## A slack bus carrying two generators


In [ ]:
import numpy as np

import dpsimpy

MW = 1e6
KV = 230e3


def slack_bus_case(name, weights, ratings=(100 * MW, 100 * MW), q_limits=None):
    n1 = dpsimpy.sp.SimNode("N1", dpsimpy.PhaseType.Single)
    n2 = dpsimpy.sp.SimNode("N2", dpsimpy.PhaseType.Single)

    generators = []
    for i in range(2):
        gen = dpsimpy.sp.ph1.SynchronGenerator("Gen" + str(i), dpsimpy.LogLevel.off)
        q_max, q_min = q_limits[i] if q_limits else (np.inf, -np.inf)
        gen.set_parameters(
            ratings[i],
            KV,
            20 * MW,
            KV,
            dpsimpy.PowerflowBusType.VD,
            5 * MW,
            q_max,
            q_min,
        )
        gen.set_base_voltage(KV)
        gen.set_slack_weight(weights[i])
        gen.connect([n1])
        generators.append(gen)

    line = dpsimpy.sp.ph1.PiLine("line", dpsimpy.LogLevel.off)
    line.set_parameters(20.0, 0.1, 0.0)
    line.set_base_voltage(KV)
    line.connect([n1, n2])

    load = dpsimpy.sp.ph1.Load("load", dpsimpy.LogLevel.off)
    load.set_parameters(100 * MW, 40 * MW, KV)
    load.modify_power_flow_bus_type(dpsimpy.PowerflowBusType.PQ)
    load.connect([n2])

    system = dpsimpy.SystemTopology(60, [n1, n2], generators + [line, load])
    sim = dpsimpy.Simulation(name, dpsimpy.LogLevel.off)
    sim.set_system(system)
    sim.set_time_step(1)
    sim.set_final_time(1)
    sim.set_domain(dpsimpy.Domain.SP)
    sim.set_solver(dpsimpy.Solver.NRP)
    sim.do_init_from_nodes_and_terminals(False)
    sim.run()

    return [
        (gen.attr("P_set").get() / MW, gen.attr("Q_set").get() / MW)
        for gen in generators
    ]

## Equal machines take equal shares

The default slack weight is 1.0 for every machine, so two identical generators
each take half of the bus total.


In [ ]:
equal = slack_bus_case("equal", [1.0, 1.0])
print(equal)

assert np.isclose(equal[0][0], equal[1][0])
assert np.isclose(equal[0][1], equal[1][1])

total_p = sum(p for p, _ in equal)
total_q = sum(q for _, q in equal)

## The weight sets the proportion

A machine with three times the weight of its neighbour takes three times the
active power, and the bus total is unchanged.


In [ ]:
weighted = slack_bus_case("weighted", [3.0, 1.0])
print(weighted)

assert np.isclose(weighted[0][0], 3.0 * weighted[1][0])
assert np.isclose(sum(p for p, _ in weighted), total_p)
assert np.isclose(sum(q for _, q in weighted), total_q)

## A weight of zero means grid following

A machine given a weight of zero keeps the set points it was configured with,
and the remaining machines absorb everything else.


In [ ]:
follower = slack_bus_case("follower", [1.0, 0.0])
print(follower)

assert np.isclose(follower[1][0], 20.0)
assert np.isclose(follower[1][1], 5.0)
assert np.isclose(sum(p for p, _ in follower), total_p)

## The rating limits the share

Once the reactive power is placed, a machine can only take what is left of its
apparent power rating. The smaller machine here would otherwise be given more
than it is rated for, so it stops on its capability circle and the larger one
picks up the rest.


In [ ]:
capped = slack_bus_case("capped", [3.0, 1.0], ratings=(60 * MW, 200 * MW))
print(capped)

apparent = np.hypot(capped[0][0], capped[0][1])
print("apparent power of the small machine:", apparent)

assert np.isclose(apparent, 60.0)
assert np.isclose(sum(p for p, _ in capped), total_p)

## Reactive power follows the reactive limits

When both machines declare finite reactive limits, each is loaded to the same
fraction of its own range, which is the convention MATPOWER uses.


In [ ]:
limits = [(80 * MW, -20 * MW), (20 * MW, -5 * MW)]
ranged = slack_bus_case("ranged", [1.0, 1.0], q_limits=limits)
print(ranged)

loading = [
    (ranged[i][1] - limits[i][1] / MW) / ((limits[i][0] - limits[i][1]) / MW)
    for i in range(2)
]
print("fraction of each reactive range:", loading)

assert np.isclose(loading[0], loading[1])
assert np.isclose(sum(q for _, q in ranged), total_q)

## Two generators on a PV bus

At a PV bus each machine keeps its own active power and only the reactive power
is shared out, so the voltage set point is held while the reactive support is
divided between the machines.


In [ ]:
n1 = dpsimpy.sp.SimNode("N1", dpsimpy.PhaseType.Single)
n2 = dpsimpy.sp.SimNode("N2", dpsimpy.PhaseType.Single)
n3 = dpsimpy.sp.SimNode("N3", dpsimpy.PhaseType.Single)

slack = dpsimpy.sp.ph1.NetworkInjection("slack", dpsimpy.LogLevel.off)
slack.set_parameters(KV)
slack.set_base_voltage(KV)
slack.modify_power_flow_bus_type(dpsimpy.PowerflowBusType.VD)
slack.connect([n1])

pv_limits = [(60 * MW, -20 * MW), (30 * MW, -10 * MW)]
pv_generators = []
for i in range(2):
    gen = dpsimpy.sp.ph1.SynchronGenerator("PvGen" + str(i), dpsimpy.LogLevel.off)
    gen.set_parameters(
        100 * MW,
        KV,
        (30 + 10 * i) * MW,
        KV,
        dpsimpy.PowerflowBusType.PV,
        0.0,
        pv_limits[i][0],
        pv_limits[i][1],
    )
    gen.set_base_voltage(KV)
    gen.connect([n2])
    pv_generators.append(gen)

line_a = dpsimpy.sp.ph1.PiLine("line_a", dpsimpy.LogLevel.off)
line_a.set_parameters(10.0, 0.05, 0.0)
line_a.set_base_voltage(KV)
line_a.connect([n1, n2])

line_b = dpsimpy.sp.ph1.PiLine("line_b", dpsimpy.LogLevel.off)
line_b.set_parameters(10.0, 0.05, 0.0)
line_b.set_base_voltage(KV)
line_b.connect([n2, n3])

pv_load = dpsimpy.sp.ph1.Load("pv_load", dpsimpy.LogLevel.off)
pv_load.set_parameters(120 * MW, 15 * MW, KV)
pv_load.modify_power_flow_bus_type(dpsimpy.PowerflowBusType.PQ)
pv_load.connect([n3])

pv_system = dpsimpy.SystemTopology(
    60, [n1, n2, n3], [slack] + pv_generators + [line_a, line_b, pv_load]
)
pv_sim = dpsimpy.Simulation("pv_bus", dpsimpy.LogLevel.off)
pv_sim.set_system(pv_system)
pv_sim.set_time_step(1)
pv_sim.set_final_time(1)
pv_sim.set_domain(dpsimpy.Domain.SP)
pv_sim.set_solver(dpsimpy.Solver.NRP)
pv_sim.do_init_from_nodes_and_terminals(False)
pv_sim.run()

pv_result = [
    (gen.attr("P_set").get() / MW, gen.attr("Q_set").get() / MW)
    for gen in pv_generators
]
print(pv_result)

Each machine keeps the active power it was given, and both are loaded to the
same fraction of their own reactive range.


In [ ]:
assert np.isclose(pv_result[0][0], 30.0)
assert np.isclose(pv_result[1][0], 40.0)

pv_loading = [
    (pv_result[i][1] - pv_limits[i][1] / MW)
    / ((pv_limits[i][0] - pv_limits[i][1]) / MW)
    for i in range(2)
]
print("fraction of each reactive range:", pv_loading)

assert np.isclose(pv_loading[0], pv_loading[1])